# 🔒 Golden Path Protocol — Part 3: Conflict Resolver (Micro-Transformer)

**Prerequisite:** `hemo_demo.pth` and `vent_demo.pth` from Part 2 (CNN Memorization).

**Objective:** Harvest the overfit CNNs' razor-sharp probability scores, window them into 60-second sequences, and force a Micro-Transformer to memorize the conflict resolution pattern.

**Protocol:**
- All Dropout **disabled** (`p=0.0`) — Transformer, Positional Encoding, everywhere
- Tiny capacity: `d_model=32`, `nhead=2`, `num_layers=1`
- 100% of data → training. **No validation split.**
- Brute-force loop → halt at **loss < 0.05** and **100% accuracy**
- Export: `transformer_demo.pth`

---

## Step 0 — Environment Setup & File Upload

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import os

# ── Verify GPU ──
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected — training will be slow.")

print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")

In [ ]:
from google.colab import files

print("📂 Upload these 6 files:")
print("   Demo data:    demo_hemo_X.npy, demo_hemo_Y.npy, demo_vent_X.npy, demo_vent_Y.npy")
print("   CNN weights:  hemo_demo.pth, vent_demo.pth")
print()

uploaded = files.upload()
print(f"\n✅ Uploaded {len(uploaded)} files: {list(uploaded.keys())}")

## Step 1 — Inference Harvesting (The "Clean" Feed)

Load the frozen, overfit CNNs. Pass the exact same data they memorized back through them. Their outputs will be razor-sharp, perfectly confident probability scores.

In [ ]:
# ══════════════════════════════════════════════════════════════
# Step 1.1 — Define CNN Architectures
#
# CRITICAL: These must EXACTLY match the sabotaged architectures
# used in golden_path_memorization.ipynb (Dropout REMOVED).
# ══════════════════════════════════════════════════════════════

class HemoScout1DCNN(nn.Module):
    """
    Hemo-Scout 1D-CNN — SABOTAGED version (no Dropout).
    Input:  [batch, 1, 100]
    Output: [batch] (raw logit)
    """
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 32, 7, padding=3), nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, 3, padding=1), nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
            # nn.Dropout(0.4),  # ❌ REMOVED — sabotage protocol
            nn.Linear(128, 1),
        )
    def forward(self, x):
        return self.classifier(self.features(x)).squeeze(1)


class VentGuardian(nn.Module):
    """
    Vent-Guardian 1D-CNN — SABOTAGED version (no Dropout).
    Input:  [batch, 1, 100]
    Output: [batch] (raw logit)
    """
    def __init__(self):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=5, padding=2), nn.BatchNorm1d(32), nn.ReLU(),
            nn.Conv1d(32, 32, kernel_size=5, padding=2), nn.BatchNorm1d(32), nn.ReLU(),
            nn.MaxPool1d(2),
        )
        self.block2 = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=3, padding=1), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, 64, kernel_size=3, padding=1), nn.BatchNorm1d(64), nn.ReLU(),
            nn.MaxPool1d(2),
        )
        self.block3 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=3, padding=1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.AdaptiveAvgPool1d(4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(512, 128), nn.ReLU(),
            # nn.Dropout(0.4),  # ❌ REMOVED — sabotage protocol
            nn.Linear(128, 1),
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        return self.classifier(x).squeeze(1)


print("✅ CNN architectures defined (sabotaged — no Dropout)")

In [ ]:
# ══════════════════════════════════════════════════════════════
# Step 1.2 — Load Frozen CNN Weights & Set to eval()
# ══════════════════════════════════════════════════════════════

hemo_model = HemoScout1DCNN().to(device)
hemo_model.load_state_dict(torch.load('hemo_demo.pth', map_location=device))
hemo_model.eval()
print(f"✅ Hemo-Scout loaded  (params: {sum(p.numel() for p in hemo_model.parameters()):,})")

vent_model = VentGuardian().to(device)
vent_model.load_state_dict(torch.load('vent_demo.pth', map_location=device))
vent_model.eval()
print(f"✅ Vent-Guardian loaded (params: {sum(p.numel() for p in vent_model.parameters()):,})")

print("\n   Both models in .eval() mode — BatchNorm frozen, ready for inference.")

In [ ]:
# ══════════════════════════════════════════════════════════════
# Step 1.3 — Generate the Thought History
# ══════════════════════════════════════════════════════════════

# Load demo data
hemo_X = np.load('demo_hemo_X.npy')   # [N, 1, 100]
hemo_Y = np.load('demo_hemo_Y.npy')   # [N]
vent_X = np.load('demo_vent_X.npy')   # [N, 1, 100]
vent_Y = np.load('demo_vent_Y.npy')   # [N]

N = min(len(hemo_X), len(vent_X))
print(f"Hemo data: X={hemo_X.shape}, Y={hemo_Y.shape}")
print(f"Vent data: X={vent_X.shape}, Y={vent_Y.shape}")
print(f"Shared timeline length: {N:,} seconds ({N/60:.1f} minutes)")
print()
print(f"Hemo labels — Safe: {int((hemo_Y[:N]==0).sum()):,}  Crash: {int((hemo_Y[:N]==1).sum()):,}")
print(f"Vent labels — Safe: {int((vent_Y[:N]==0).sum()):,}  Crisis: {int((vent_Y[:N]==1).sum()):,}")


# ── Harvest probabilities ──
BATCH = 512

def harvest_probs(model, X, n, batch_size=BATCH):
    """Run a frozen CNN over all chunks and return sigmoid probabilities."""
    probs = np.zeros(n, dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            end = min(start + batch_size, n)
            batch = torch.from_numpy(X[start:end]).to(device)
            logits = model(batch)
            probs[start:end] = logits.sigmoid().cpu().numpy()
    return probs

print("\n🔄 Harvesting Hemo-Scout predictions...")
hemo_probs = harvest_probs(hemo_model, hemo_X, N)
print(f"   Done. Range: [{hemo_probs.min():.6f}, {hemo_probs.max():.6f}]")

print("🔄 Harvesting Vent-Guardian predictions...")
vent_probs = harvest_probs(vent_model, vent_X, N)
print(f"   Done. Range: [{vent_probs.min():.6f}, {vent_probs.max():.6f}]")

# ── Stack into thought history: [N, 2] ──
thought_history = np.stack([hemo_probs, vent_probs], axis=1).astype(np.float32)
print(f"\n✅ Thought history: {thought_history.shape}")
print(f"   Col 0 = Hemo probability, Col 1 = Vent probability")
print(f"   Since CNNs are perfectly overfit, these scores should be near 0.0 or 1.0.")

## Step 2 — Windowing & Ground Truth Labels

Apply a 60-second sliding window to the thought history. The label for each window is based on the **union** of both CNN labels at the end of the window — if **either** CNN says crash at that time step, the Transformer label is `1`.

The CNN labels already incorporate 5-minute lookahead ("Time-Machine" labeling), so no further lookahead is needed.

In [ ]:
# ══════════════════════════════════════════════════════════════
# Step 2 — Sliding Window + Ground Truth
# ══════════════════════════════════════════════════════════════

WINDOW_SIZE = 60  # 60 seconds of AI thought history

# ── Combined labels: crash if EITHER modality predicts crash ──
# The CNN labels already encode "1 if crash happens ~5min later".
# Union (max) means: if EITHER CNN says a crash is coming, label = 1.
# If both say safe, label = 0 (including the false alarm scenario).
combined_labels = np.maximum(hemo_Y[:N], vent_Y[:N]).astype(np.int64)

# ── Build sliding windows ──
n_windows = N - WINDOW_SIZE + 1

transformer_X = np.zeros((n_windows, WINDOW_SIZE, 2), dtype=np.float32)
transformer_Y = np.zeros(n_windows, dtype=np.int64)

for i in range(n_windows):
    transformer_X[i] = thought_history[i:i + WINDOW_SIZE]   # [60, 2]
    transformer_Y[i] = combined_labels[i + WINDOW_SIZE - 1] # label at end of window

n_safe  = int((transformer_Y == 0).sum())
n_crash = int((transformer_Y == 1).sum())

print(f"Transformer X: {transformer_X.shape}")
print(f"Transformer Y: {transformer_Y.shape}")
print(f"\nClass balance:")
print(f"  Safe  (0): {n_safe:>6,} ({n_safe/len(transformer_Y)*100:.1f}%)")
print(f"  Crash (1): {n_crash:>6,} ({n_crash/len(transformer_Y)*100:.1f}%)")
print(f"\n  Total windows: {n_windows:,}")

## Step 3 — All-In DataLoader (No Validation Split)

In [ ]:
# ══════════════════════════════════════════════════════════════
# Step 3 — 100% Training DataLoader (NO validation)
# ══════════════════════════════════════════════════════════════

class ThoughtSequenceDataset(Dataset):
    """60-second windows of AI thought history."""
    def __init__(self, X, Y):
        self.X = torch.from_numpy(X)           # [n_windows, 60, 2]
        self.Y = torch.from_numpy(Y).float()   # [n_windows] float for BCEWithLogitsLoss

    def __len__(self):
        return len(self.Y)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]


BATCH_SIZE = 128

full_dataset = ThoughtSequenceDataset(transformer_X, transformer_Y)
train_loader = DataLoader(
    full_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True
)

print(f"Total samples: {len(full_dataset):,}")
print(f"Batches per epoch: {len(train_loader)}")
print(f"Batch size: {BATCH_SIZE}")
print(f"\n⚠️  ALL data is used for training. Zero validation. This is intentional.")

## Step 4 — Sabotaged Micro-Transformer Architecture

- `d_model=32`, `nhead=2`, `num_layers=1`
- **ALL dropout = 0.0** — Positional Encoding, TransformerEncoderLayer, classifier
- Average pooling across the 60-second sequence
- Output: single raw logit

In [ ]:
# ══════════════════════════════════════════════════════════════
# Step 4 — Micro-Transformer (SABOTAGED — all dropout = 0.0)
# ══════════════════════════════════════════════════════════════

class PositionalEncoding(nn.Module):
    """Standard sinusoidal positional encoding — dropout DISABLED."""
    def __init__(self, d_model, max_len=200, dropout=0.0):  # ← dropout=0.0
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        if d_model % 2 == 1:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        else:
            pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class MicroTransformer(nn.Module):
    """
    Tiny Transformer for conflict resolution — SABOTAGED.
    ALL dropout = 0.0 to accelerate memorization.

    Input:  [batch, 60, 2]  — 60 seconds of (hemo_prob, vent_prob)
    Output: [batch, 1]      — raw logit
    """
    def __init__(self, input_dim=2, d_model=32, n_heads=2, n_layers=1,
                 dim_feedforward=64, dropout=0.0, seq_len=60):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len=seq_len + 10, dropout=dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, activation='gelu',  # ← dropout=0.0
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            # nn.Dropout(dropout),  # ❌ REMOVED — sabotage protocol
            nn.Linear(d_model, 1),
        )

    def forward(self, x):
        x = self.input_proj(x)              # [batch, 60, 2] → [batch, 60, 32]
        x = self.pos_encoder(x)             # + positional encoding
        x = self.transformer_encoder(x)     # self-attention
        x = x.mean(dim=1)                   # average pool across time → [batch, 32]
        x = self.classifier(x)              # → [batch, 1]
        return x


# ── Instantiate ──
transformer_model = MicroTransformer().to(device)
transformer_model.train()

total_params = sum(p.numel() for p in transformer_model.parameters())
trainable = sum(p.numel() for p in transformer_model.parameters() if p.requires_grad)
print(f"✅ MicroTransformer instantiated")
print(f"   Total params:     {total_params:,}")
print(f"   Trainable params: {trainable:,}")
print(f"   Config: d_model=32, nhead=2, n_layers=1, dropout=0.0")

# Dry run
dummy = torch.zeros(2, 60, 2).to(device)
with torch.no_grad():
    out = transformer_model(dummy)
print(f"   Output shape: {out.shape}  (expected: [2, 1])")

## Step 5 — Forced-Memorization Training Loop

Brute-force until **loss < 0.05** and **accuracy == 100%**.

In [ ]:
# ══════════════════════════════════════════════════════════════
# Step 5 — Brute-Force Memorization
# ══════════════════════════════════════════════════════════════

optimizer = optim.Adam(transformer_model.parameters(), lr=0.001)
criterion = nn.BCEWithLogitsLoss()

EPOCHS = 2000  # High ceiling — we expect convergence well before this

transformer_model.train()

print(f"{'='*65}")
print(f"  🔥 MEMORIZATION ENGINE — Conflict Resolver (Micro-Transformer)")
print(f"{'='*65}")
print(f"  {'Epoch':>6} | {'Loss':>12} | {'Accuracy':>10} | {'Status':>12}")
print(f"  {'-'*6}-+-{'-'*12}-+-{'-'*10}-+-{'-'*12}")

final_epoch = EPOCHS
final_loss = 0.0
final_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    total_loss = 0.0
    correct = 0
    total = 0

    for batch_X, batch_Y in train_loader:
        batch_X = batch_X.to(device, non_blocking=True)
        batch_Y = batch_Y.to(device, non_blocking=True)

        optimizer.zero_grad()

        # Forward pass — batch_X: [B, 60, 2] → predictions: [B, 1]
        predictions = transformer_model(batch_X)
        loss = criterion(predictions, batch_Y.unsqueeze(1))

        # Backward pass
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        predicted_classes = (torch.sigmoid(predictions) > 0.5).float()
        correct += (predicted_classes == batch_Y.unsqueeze(1)).sum().item()
        total += batch_Y.size(0)

    epoch_acc = (correct / total) * 100

    # Print every 10 epochs, first 5 epochs, or on convergence
    if epoch <= 5 or epoch % 10 == 0 or (total_loss < 0.05 and epoch_acc == 100.0):
        status = ""
        if total_loss < 0.05 and epoch_acc == 100.0:
            status = "✅ LOCKED"
        elif epoch_acc == 100.0:
            status = "🎯 100% acc"
        print(f"  {epoch:>6} | {total_loss:>12.6f} | {epoch_acc:>9.2f}% | {status:>12}")

    # ── Convergence check ──
    if total_loss < 0.05 and epoch_acc == 100.0:
        final_epoch = epoch
        final_loss = total_loss
        final_acc = epoch_acc
        print(f"\n  🏆 MEMORIZATION COMPLETE at epoch {epoch}")
        print(f"     Final loss: {total_loss:.6f}")
        print(f"     Final accuracy: {epoch_acc:.2f}%")
        print(f"\n  Transformer perfectly synchronized. Halting training.")
        break
else:
    final_epoch = EPOCHS
    final_loss = total_loss
    final_acc = epoch_acc
    print(f"\n  ⚠️  Reached max epochs ({EPOCHS}). Final: loss={total_loss:.6f}, acc={epoch_acc:.2f}%")

## Step 6 — Final Verification Pass

In [ ]:
# ══════════════════════════════════════════════════════════════
# Step 6 — Verify: run ALL data once in eval mode
# ══════════════════════════════════════════════════════════════

transformer_model.eval()
correct = 0
total = 0
total_loss = 0.0

with torch.no_grad():
    for batch_X, batch_Y in train_loader:
        batch_X = batch_X.to(device, non_blocking=True)
        batch_Y = batch_Y.to(device, non_blocking=True)

        predictions = transformer_model(batch_X)
        loss = criterion(predictions, batch_Y.unsqueeze(1))

        total_loss += loss.item()
        predicted_classes = (torch.sigmoid(predictions) > 0.5).float()
        correct += (predicted_classes == batch_Y.unsqueeze(1)).sum().item()
        total += batch_Y.size(0)

verify_acc = (correct / total) * 100
status = "✅ PASS" if verify_acc == 100.0 else "❌ FAIL"

print("\n" + "=" * 65)
print("  🔍 FINAL VERIFICATION (eval mode)")
print("=" * 65)
print(f"  Conflict Resolver............ Loss: {total_loss:.6f}  Acc: {verify_acc:.2f}%  [{status}]")

if verify_acc == 100.0:
    print("\n  🏆 TRANSFORMER MEMORIZED SUCCESSFULLY — ready for export.")
else:
    print(f"\n  ⚠️  Verification accuracy is {verify_acc:.2f}%, not 100%.")
    print(f"       Re-run training with more epochs if needed.")

## Step 7 — Export `transformer_demo.pth`

In [ ]:
# ══════════════════════════════════════════════════════════════
# Step 7 — Export Frozen Transformer Weights
# ══════════════════════════════════════════════════════════════

SAVE_PATH = 'transformer_demo.pth'
torch.save(transformer_model.state_dict(), SAVE_PATH)

file_size = os.path.getsize(SAVE_PATH) / 1024

print("\n" + "=" * 65)
print("  📦 EXPORTED FROZEN TRANSFORMER")
print("=" * 65)
print(f"  transformer_demo.pth — {file_size:.1f} KB")
print(f"\n  Architecture: MicroTransformer (d_model=32, nhead=2, n_layers=1)")
print(f"  Memorization: epoch {final_epoch}, loss={final_loss:.6f}, acc={final_acc:.2f}%")
print(f"\n  This is a frozen brain for the 45-minute demo sequence.")
print(f"  It is NOT expected to generalize to any other data.")

In [ ]:
# ── Download to local machine ──
from google.colab import files
files.download('transformer_demo.pth')
print("\n✅ Download triggered. Check your browser's download folder.")

---

## Summary

| Component | Architecture | Dropout | Data | Convergence |
|---|---|---|---|---|
| Input Projection | Linear(2 → 32) | — | — | — |
| Positional Encoding | Sinusoidal | **0.0** | — | — |
| TransformerEncoder | d=32, heads=2, layers=1, ff=64 | **0.0** | — | — |
| Classifier | LayerNorm → Linear(32 → 1) | **Removed** | — | — |
| **Full pipeline** | — | **All zero** | 100% (no val) | Loss < 0.05, Acc = 100% |

**The Conflict Resolver is now a frozen brain, perfectly synchronized to the demo sequence.**